## Working with data

In [6]:
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)
# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

100%|███████████████████████████████████████████████████████████| 26.4M/26.4M [01:31<00:00, 289kB/s]
100%|███████████████████████████████████████████████████████████| 29.5k/29.5k [00:00<00:00, 168kB/s]
100%|███████████████████████████████████████████████████████████| 4.42M/4.42M [00:20<00:00, 214kB/s]
100%|██████████████████████████████████████████████████████████| 5.15k/5.15k [00:00<00:00, 7.06MB/s]


In [3]:
BATCH_SIZE = 64
# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE)

for X, y in test_dataloader:
    # Display image
    print(X.shape, y.shape) # [N, C, H, W]
    break

torch.Size([64, 1, 28, 28]) torch.Size([64])


## Creating Models

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [5]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


## Optimizing the Model Parameters

In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [8]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [9]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [10]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.301126  [   64/60000]
loss: 0.562486  [ 6464/60000]
loss: 0.391398  [12864/60000]
loss: 0.505347  [19264/60000]
loss: 0.456793  [25664/60000]
loss: 0.440676  [32064/60000]
loss: 0.376414  [38464/60000]
loss: 0.535482  [44864/60000]
loss: 0.469719  [51264/60000]
loss: 0.494131  [57664/60000]
Test Error: 
 Accuracy: 84.7%, Avg loss: 0.425991 

Epoch 2
-------------------------------
loss: 0.280377  [   64/60000]
loss: 0.362715  [ 6464/60000]
loss: 0.266719  [12864/60000]
loss: 0.390544  [19264/60000]
loss: 0.410196  [25664/60000]
loss: 0.388272  [32064/60000]
loss: 0.320935  [38464/60000]
loss: 0.489893  [44864/60000]
loss: 0.397666  [51264/60000]
loss: 0.488183  [57664/60000]
Test Error: 
 Accuracy: 85.0%, Avg loss: 0.402374 

Epoch 3
-------------------------------
loss: 0.229926  [   64/60000]
loss: 0.325601  [ 6464/60000]
loss: 0.242561  [12864/60000]
loss: 0.367256  [19264/60000]
loss: 0.389085  [25664/60000]
loss: 0.360182  [32064/600

## Saving Models

In [11]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


## Loading Models

In [12]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [13]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]
model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"
